In [ ]:
import yaml
import torch
import torchmetrics
import torch.nn as nn
from collections import defaultdict
from functools import partial
from torchview import draw_graph

from cfdna.preprocessing.transforms import build_transform_pipeline
from cfdna.training.utils import get_dataloaders as _get_dataloaders, SelfTargetLoader
from cfdna.training.trainer import (
    train,
    compute_best_roc_data,
    plot_training_progress,
    NegReconMSE,
)


if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'using device: {device}')


with open('../confs/thesis.yaml', 'r') as f:
    config = yaml.safe_load(f)

OUTPUT_DIR = f'../{config["data"]["training_output_dir"]}'
TRAIN_SIZE = config['training']['train_size']
VALID_SIZE = config['training']['valid_size']
BATCH_SIZE = config['training']['batch_size']
SEED = 42

history = defaultdict(str)


def get_dataloaders(
    do_standardization=False,
    do_sum_normalization=False,
    do_log_transform=False,
    is_tiny=False,
    slice_params=None,
    suffix='downsampled',
    only_positive=False,
):
    """Notebook wrapper around cfdna.training.utils.get_dataloaders.

    Translates the boolean flags used in the notebook cells into a
    transforms list that build_transform_pipeline understands.
    """
    transform_configs = []
    if slice_params is not None:
        transform_configs.append({'name': 'slice', 'params': slice_params})
    if do_sum_normalization:
        transform_configs.append({'name': 'sum_normalization'})
    if do_log_transform:
        transform_configs.append({'name': 'log_transform'})
    if do_standardization:
        transform_configs.append({'name': 'standardization'})

    transform_fn = build_transform_pipeline(transform_configs) if transform_configs else None

    return _get_dataloaders(
        output_dir=OUTPUT_DIR,
        transform_fn=transform_fn,
        needs_standardization=do_standardization,
        train_size=TRAIN_SIZE,
        valid_size=VALID_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
        suffix=suffix,
        is_tiny=is_tiny,
        only_positive=only_positive,
    )


def build_model(m, seed=SEED):
    torch.manual_seed(seed)
    return m().to(device)


def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

In [ ]:
# SimpleCNNModel:

# SimpleCNNUnit (n times):
#  - Convolution layers (2d)
#  - LeakyReLU
#  - MaxPool (2d)
# AdaptiveAvgPool2d (global average pooling layer)
# Flatten
# Linear (head -> outputs logits)


class SimpleCNNUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.1):
        super().__init__()
        DefaultConv2d = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=True)
        self.layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels, stride=stride),
            # nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            DefaultConv2d(out_channels, out_channels),
            # nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=2),
            # nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.layers(x)


class SimpleCNNModel(nn.Module):
    def __init__(self, base_channels=16, dropout=0.10, num_layers=2):
        super().__init__()
        layers = [SimpleCNNUnit(1, base_channels)]

        prev_c = base_channels
        for c in [base_channels * 2, base_channels * 4]:
            layers.append(SimpleCNNUnit(prev_c, c))
            prev_c = c

        layers += [
            nn.AdaptiveAvgPool2d((1, 8)),
            nn.Flatten(),
            nn.Linear(prev_c, 1),
        ]
        self.cnn = nn.Sequential(*layers)

    def forward(self, x):
        return self.cnn(x).squeeze(-1)


n_epochs = 20
tag = 'cnn_1_2000'

model = build_model(SimpleCNNModel)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = (
    nn.BCEWithLogitsLoss()
)  # outputs logits (-infinity <-> +infinity) -> use sigmoid to get probs
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_cnn = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)

history[tag] = {**roc_data, **history_cnn}

In [ ]:
# TweakedCNNModel:


class TweakedCNNUnit(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.1, pool_ks=(2, 4)):
        super().__init__()
        DefaultConv2d = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            DefaultConv2d(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=pool_ks),
            # nn.Dropout2d(dropout)
        )

    def forward(self, x):
        return self.layers(x)


class TweakedCNNModel(nn.Module):
    def __init__(self, base_channels=32, dropout=0.10, num_layers=2):
        super().__init__()
        C = base_channels
        self.cnn = nn.Sequential(
            TweakedCNNUnit(1, C, pool_ks=(2, 4)),
            TweakedCNNUnit(C, C * 2, pool_ks=(2, 4)),
            TweakedCNNUnit(C * 2, C * 4, pool_ks=(1, 5)),
            TweakedCNNUnit(C * 4, C * 8, pool_ks=(1, 5)),
            nn.AdaptiveAvgPool2d((1, 4)),
            nn.Flatten(),
            nn.Linear(C * 8 * 4, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.cnn(x).squeeze(-1)


n_epochs = 20
tag = 'cnn_1_2000_tweak'

model = build_model(TweakedCNNModel)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = (
    nn.BCEWithLogitsLoss()
)  # outputs logits (-infinity <-> +infinity) -> use sigmoid to get probs
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_cnn = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    # scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)

history[tag] = {**roc_data, **history_cnn}

In [ ]:
## MLPOnRelativeMidpointsModel (lengths are collapsed 300x2000 -> 1x2000):

# SimpleMLPUnit (n times):
#  - Linear
#  - ReLU
# Linear (head -> outputs logits)


class SimpleMLPUnit(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(in_features, out_features), nn.ReLU())

    def forward(self, x):
        return self.layers(x)


class MLPOnRelativeMidpointsModel(nn.Module):
    # def __init__(self, n_inputs=2000, n_neurons=[2048, 1024, 1024, 512]):
    def __init__(self, n_inputs=2000, n_neurons=[1024, 1024, 512]):
        super().__init__()
        layers = [
            SimpleMLPUnit(n_in, n_out) for n_in, n_out in zip([n_inputs] + n_neurons, n_neurons)
        ] + [nn.Linear(n_neurons[-1], 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, X):
        # X: (B, 1, 300, 2000)
        X = X.sum(dim=2)  # (B, 1, 300, 2000) -> (B, 1, 2000)
        X = X.squeeze(1)  # (B, 1, 2000) -> (B, 2000)
        return self.mlp(X).squeeze(1)


n_epochs = 100
tag = 'mlp_1_2000'

model = build_model(MLPOnRelativeMidpointsModel)
model.apply(use_he_init)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = (
    nn.BCEWithLogitsLoss()
)  # outputs logits (-infinity <-> +infinity) -> use sigmoid to get probs
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_relative_midpoints = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)

history[tag] = {**roc_data, **history_relative_midpoints}

In [ ]:
## MLPOnFragmentLengthsModel (relative midpoints are collapsed 300x2000 -> 300x1):

# SimpleMLPUnit (n times):
#  - Linear
#  - ReLU
# Linear (head -> outputs logits)
class MLPOnFragmentLengthsModel(nn.Module):
    # def __init__(self, n_inputs=300, n_neurons=[2048, 1024, 1024, 512]):
    def __init__(self, n_inputs=300, n_neurons=[1024, 1024, 512]):
        super().__init__()
        layers = [
            SimpleMLPUnit(n_in, n_out) for n_in, n_out in zip([n_inputs] + n_neurons, n_neurons)
        ] + [nn.Linear(n_neurons[-1], 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, X):
        X = X.sum(dim=3)
        # X = fragle_transforms(X)
        X = X.squeeze(1)
        return self.mlp(X).squeeze(1)


n_epochs = 100
tag = 'mlp_300_1'

model = build_model(MLPOnFragmentLengthsModel)
model.apply(use_he_init)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = (
    nn.BCEWithLogitsLoss()
)  # outputs logits (-infinity <-> +infinity) -> use sigmoid to get probs
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_relative_midpoints = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)

history[tag] = {**roc_data, **history_relative_midpoints}

In [ ]:
## MLPMultipleInputsModel (both relative midpoints and fragments are collapsed and given as inputs to the MLP):

# SimpleMLPUnit (n times):
#  - Linear
#  - ReLU
# Linear (head -> outputs logits)
class MLPMultipleInputsModel(nn.Module):
    # def __init__(self, n_inputs=2000+300, n_neurons=[2048, 1024, 1024, 512]):
    def __init__(self, n_inputs=2000 + 300, n_neurons=[1024, 1024, 512]):
        super().__init__()
        layers = [
            SimpleMLPUnit(n_in, n_out) for n_in, n_out in zip([n_inputs] + n_neurons, n_neurons)
        ] + [nn.Linear(n_neurons[-1], 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, X):
        frags = X.sum(dim=3)
        relative_midpoints = X.sum(dim=2)
        X = torch.concat([frags, relative_midpoints], dim=2)
        X = X.squeeze(1)
        return self.mlp(X).squeeze(1)


n_epochs = 100
tag = 'mlp_multiple_inputs'

model = build_model(MLPMultipleInputsModel)
model.apply(use_he_init)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = (
    nn.BCEWithLogitsLoss()
)  # outputs logits (-infinity <-> +infinity) -> use sigmoid to get probs
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_relative_midpoints = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    patience=20,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)

history[tag] = {**roc_data, **history_relative_midpoints}

In [ ]:
# RebinnedCNNModel - designed for 46×2000 input


class RebinnedCNNUnit(nn.Module):
    def __init__(self, in_channels, out_channels, pool_ks=(2, 4)):
        super().__init__()
        DefaultConv2d = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            DefaultConv2d(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(kernel_size=pool_ks),
        )

    def forward(self, x):
        return self.layers(x)


class RebinnedCNNModel(nn.Module):
    def __init__(self, base_channels=32):
        super().__init__()
        C = base_channels
        self.cnn = nn.Sequential(
            RebinnedCNNUnit(1, C, pool_ks=(2, 4)),
            RebinnedCNNUnit(C, C * 2, pool_ks=(2, 4)),
            # RebinnedCNNUnit(C * 2, C * 4, pool_ks=(1, 5)),
            # RebinnedCNNUnit(C * 4, C * 8, pool_ks=(1, 5)),
            nn.AdaptiveAvgPool2d((1, 4)),
            nn.Flatten(),
            nn.Linear(C * 2 * 4, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.cnn(x).squeeze(-1)


n_epochs = 20
tag = 'cnn_rebinned_46x2000'

model = build_model(RebinnedCNNModel)
model.apply(use_he_init)

optimizer = torch.optim.Adam(model.parameters())
binaryxentropy = nn.BCEWithLogitsLoss()
binary_auc = torchmetrics.classification.BinaryAUROC().to(device)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.1
)

train_loader, valid_loader, _ = get_dataloaders(
    suffix='rebinned',
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

history_cnn = train(
    model,
    optimizer,
    binaryxentropy,
    binary_auc,
    train_loader,
    valid_loader,
    n_epochs,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

binary_roc = torchmetrics.classification.BinaryROC().to(device)
roc_data = compute_best_roc_data(model, valid_loader, roc_metric=binary_roc, device=device)
history[tag] = {**roc_data, **history_cnn}

In [ ]:
from cfdna.models.vae import VAEModel, vae_loss

n_epochs = 20
tag = 'vae_rebinned'
beta = 1.0

train_loader, valid_loader, _ = get_dataloaders(
    suffix='rebinned',
    only_positive=True,
    # do_standardization=True,
    # do_sum_normalization=True,
    # do_log_transform=True,
    # is_tiny=True,
    # slice_params={'ymin': 130, 'ymax': 200, 'xmin': 600, 'xmax': 1400},
)

# get input dims from a sample batch
sample_x, _ = next(iter(train_loader))
input_height, input_width = sample_x.shape[2], sample_x.shape[3]

torch.manual_seed(SEED)
model = VAEModel(input_height=input_height, input_width=input_width).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.1
)

metric = NegReconMSE().to(device)


def loss_fn(vae_output, target):
    return vae_loss(vae_output.reconstruction, target, vae_output.mu, vae_output.logvar, beta=beta)


vae_train_loader = SelfTargetLoader(train_loader)
vae_valid_loader = SelfTargetLoader(valid_loader)

history_vae = train(
    model,
    optimizer,
    loss_fn,
    metric,
    vae_train_loader,
    vae_valid_loader,
    n_epochs,
    patience=10,
    scheduler=perf_scheduler,
    checkpoint_path=f'{tag}.pt',
    device=device,
)

history[tag] = history_vae

In [ ]:
plot_training_progress(history)

In [ ]:
from cfdna.models import get_model
# from cfdna.models.mlp_model import SimpleMLPUnit
# from cfdna.models.cnn_model import RebinnedCNNUnit
# from cfdna.models.vae import EncoderBlock, DecoderBlock


train_loader, valid_loader, _ = get_dataloaders()

# input_size = None
for X_batch, y_batch in SelfTargetLoader(train_loader):
    input_size = X_batch.shape
    break

# get_dataloaders()
model = get_model('mlp')
# model = build_model(partial(SimpleMLPUnit, in_features=2000+300, out_features=1024))  # input_size (1, 2300)

# get_dataloaders(suffix='rebinned')
# model = get_model('cnn')
# model = build_model(partial(RebinnedCNNUnit, in_channels=1, out_channels=32))  # input_size (1, 1, 32, 192)

# get_dataloaders(suffix='rebinned', only_positive=True)
# model = get_model('vae', input_height=input_size[2], input_width=input_size[3],)
# model = build_model(partial(EncoderBlock, in_channels=1, out_channels=32))  # input_size (1, 1, 32, 192)
# model = build_model(partial(DecoderBlock, 128, 64))  # input_size (1, 1, 128, 64)


model_graph = draw_graph(
    model=model,
    input_size=input_size,
    device='meta',
    expand_nested=True,
    save_graph=True,
    filename='architectures/mlp_multiple_inputs_300x2000',
)
model_graph.visual_graph

In [ ]:
print(
    f'Total num of learnable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}'
)